# Working Backwards Experiments

This notebook generates synthetic latency samples for exploration and exports summary metrics to `docs/notebooks/artifacts/latency_summary.json`.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import random


In [2]:
NOTEBOOK_PATH = Path("docs/notebooks/working_backwards_experiments.ipynb")
ARTIFACT_PATH = Path("docs/notebooks/artifacts/latency_summary.json")
SEED = 20240919
SAMPLE_SIZE = 500

random.seed(SEED)
latency_samples = [max(0.0, round(random.gauss(320, 45), 2)) for _ in range(SAMPLE_SIZE)]
latency_samples[:5]

[309.04, 294.35, 282.19, 325.45, 323.86]

In [3]:
sorted_samples = sorted(latency_samples)

def percentile(sorted_data, p):
    if len(sorted_data) == 1:
        return float(sorted_data[0])
    if not 0 <= p <= 100:
        raise ValueError("percentile must be between 0 and 100")

    k = (len(sorted_data) - 1) * (p / 100)
    f = int(k)
    c = min(f + 1, len(sorted_data) - 1)
    if f == c:
        return float(sorted_data[f])
    remainder = k - f
    return float(sorted_data[f] + (sorted_data[c] - sorted_data[f]) * remainder)

metrics = {
    "notebook": str(NOTEBOOK_PATH),
    "seed": SEED,
    "runs": len(latency_samples),
    "latency_ms": {
        "mean": round(sum(latency_samples) / len(latency_samples), 3),
        "median": round(sorted_samples[len(sorted_samples) // 2], 3),
        "p75": round(percentile(sorted_samples, 75), 3),
        "p90": round(percentile(sorted_samples, 90), 3),
        "p95": round(percentile(sorted_samples, 95), 3),
        "min": round(sorted_samples[0], 3),
        "max": round(sorted_samples[-1], 3)
    }
}
metrics

{'notebook': 'docs/notebooks/working_backwards_experiments.ipynb', 'seed': 20240919, 'runs': 500, 'latency_ms': {'mean': 319.958, 'median': 319.57, 'p75': 349.695, 'p90': 374.634, 'p95': 389.272, 'min': 201.79, 'max': 467.41}}

In [4]:
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATH.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
json.loads(ARTIFACT_PATH.read_text(encoding="utf-8"))


{'notebook': 'docs/notebooks/working_backwards_experiments.ipynb', 'seed': 20240919, 'runs': 500, 'latency_ms': {'mean': 319.958, 'median': 319.57, 'p75': 349.695, 'p90': 374.634, 'p95': 389.272, 'min': 201.79, 'max': 467.41}}